In [1]:
"""
Train/Validation/Test Split Script for Brain Tumor MRI Dataset
Author: Your Name
Project: 10-Class Brain Tumor Detection

This script splits preprocessed images into:
- Training set: 70%
- Validation set: 15%
- Test set: 15%

Features:
- Stratified split (maintains class distribution)
- Reproducible (fixed random seed)
- Copies files to appropriate directories
- Generates split statistics
"""

import os
import shutil
import random
import json
from pathlib import Path
from collections import defaultdict
from datetime import datetime
import numpy as np


class DatasetSplitter:
    """
    Split preprocessed brain MRI dataset into train/val/test sets
    """

    def __init__(self, input_dir, output_dir, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15, random_seed=42):
        """
        Initialize the splitter

        Args:
            input_dir (str): Path to preprocessed dataset
            output_dir (str): Path to output directory for splits
            train_ratio (float): Proportion for training set
            val_ratio (float): Proportion for validation set
            test_ratio (float): Proportion for test set
            random_seed (int): Random seed for reproducibility
        """
        self.input_dir = Path(input_dir)
        self.output_dir = Path(output_dir)
        self.train_ratio = train_ratio
        self.val_ratio = val_ratio
        self.test_ratio = test_ratio
        self.random_seed = random_seed

        # Validate ratios
        total_ratio = train_ratio + val_ratio + test_ratio
        if not np.isclose(total_ratio, 1.0):
            raise ValueError(f"Ratios must sum to 1.0, got {total_ratio}")

        # Define the 10 classes (LOCKED 🔒)
        self.classes = [
            'Astrocytoma',
            'Ependymoma',
            'Glioblastoma',
            'Germinoma',
            'Medulloblastoma',
            'Meningioma',
            'Oligodendroglioma',
            'Pituitary',
            'Schwannoma',
            'No_Tumor'
        ]

        self.split_stats = {
            'train': defaultdict(int),
            'val': defaultdict(int),
            'test': defaultdict(int)
        }

        # Set random seed for reproducibility
        random.seed(self.random_seed)
        np.random.seed(self.random_seed)

    def create_directory_structure(self):
        """Create output directory structure"""
        for split in ['train', 'val', 'test']:
            split_dir = self.output_dir / split
            split_dir.mkdir(parents=True, exist_ok=True)

            for class_name in self.classes:
                class_dir = split_dir / class_name
                class_dir.mkdir(exist_ok=True)

    def get_class_images(self, class_name):
        """
        Get all image files for a specific class

        Args:
            class_name (str): Name of the class

        Returns:
            list: List of image file paths
        """
        class_dir = self.input_dir / class_name

        if not class_dir.exists():
            print(f"⚠️  Warning: {class_dir} does not exist")
            return []

        # Get all image files
        image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff']
        image_files = []
        for ext in image_extensions:
            image_files.extend(list(class_dir.glob(f'*{ext}')))
            image_files.extend(list(class_dir.glob(f'*{ext.upper()}')))

        return image_files

    def stratified_split(self, images):
        """
        Perform stratified split on images

        Args:
            images (list): List of image file paths

        Returns:
            tuple: (train_images, val_images, test_images)
        """
        # Shuffle images
        images = list(images)
        random.shuffle(images)

        n_total = len(images)
        n_train = int(n_total * self.train_ratio)
        n_val = int(n_total * self.val_ratio)

        # Split
        train_images = images[:n_train]
        val_images = images[n_train:n_train + n_val]
        test_images = images[n_train + n_val:]

        return train_images, val_images, test_images

    def copy_images(self, image_list, split_name, class_name):
        """
        Copy images to the appropriate split directory

        Args:
            image_list (list): List of image paths to copy
            split_name (str): 'train', 'val', or 'test'
            class_name (str): Name of the class
        """
        dest_dir = self.output_dir / split_name / class_name

        for img_path in image_list:
            dest_path = dest_dir / img_path.name
            shutil.copy2(img_path, dest_path)
            self.split_stats[split_name][class_name] += 1

    def split_dataset(self):
        """
        Perform the complete train/val/test split
        """
        print("=" * 80)
        print("🔀 TRAIN/VAL/TEST SPLIT")
        print("=" * 80)
        print(f"Input directory: {self.input_dir}")
        print(f"Output directory: {self.output_dir}")
        print(f"Split ratios - Train: {self.train_ratio*100:.0f}% | Val: {self.val_ratio*100:.0f}% | Test: {self.test_ratio*100:.0f}%")
        print(f"Random seed: {self.random_seed}")
        print("=" * 80)

        # Create directory structure
        print("\n📁 Creating directory structure...")
        self.create_directory_structure()

        # Process each class
        total_stats = {'train': 0, 'val': 0, 'test': 0}

        for class_name in self.classes:
            print(f"\n📊 Processing class: {class_name}")

            # Get all images for this class
            images = self.get_class_images(class_name)

            if not images:
                print(f"  ⚠️  No images found, skipping...")
                continue

            print(f"  Total images: {len(images)}")

            # Perform stratified split
            train_imgs, val_imgs, test_imgs = self.stratified_split(images)

            # Copy images to respective directories
            print(f"  Copying to train: {len(train_imgs)} images")
            self.copy_images(train_imgs, 'train', class_name)

            print(f"  Copying to val: {len(val_imgs)} images")
            self.copy_images(val_imgs, 'val', class_name)

            print(f"  Copying to test: {len(test_imgs)} images")
            self.copy_images(test_imgs, 'test', class_name)

            total_stats['train'] += len(train_imgs)
            total_stats['val'] += len(val_imgs)
            total_stats['test'] += len(test_imgs)

        # Print statistics
        self.print_statistics(total_stats)

        # Save statistics
        self.save_statistics(total_stats)

    def print_statistics(self, total_stats):
        """Print split statistics"""
        print("\n" + "=" * 80)
        print("📊 SPLIT STATISTICS")
        print("=" * 80)

        print(f"\n{'Class':<20} {'Train':<10} {'Val':<10} {'Test':<10} {'Total':<10}")
        print("-" * 70)

        for class_name in self.classes:
            train_count = self.split_stats['train'][class_name]
            val_count = self.split_stats['val'][class_name]
            test_count = self.split_stats['test'][class_name]
            total = train_count + val_count + test_count

            print(f"{class_name:<20} {train_count:<10} {val_count:<10} {test_count:<10} {total:<10}")

        print("-" * 70)
        print(f"{'TOTAL':<20} {total_stats['train']:<10} {total_stats['val']:<10} {total_stats['test']:<10} {sum(total_stats.values()):<10}")

        print("\n📈 Actual Split Ratios:")
        total_images = sum(total_stats.values())
        if total_images > 0:
            print(f"  Train: {total_stats['train']/total_images*100:.2f}%")
            print(f"  Val:   {total_stats['val']/total_images*100:.2f}%")
            print(f"  Test:  {total_stats['test']/total_images*100:.2f}%")

        print("=" * 80)

    def save_statistics(self, total_stats):
        """Save statistics to JSON file"""
        stats_file = self.output_dir / 'split_statistics.json'

        stats_data = {
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'input_directory': str(self.input_dir),
            'output_directory': str(self.output_dir),
            'random_seed': self.random_seed,
            'split_ratios': {
                'train': self.train_ratio,
                'val': self.val_ratio,
                'test': self.test_ratio
            },
            'class_statistics': {
                class_name: {
                    'train': self.split_stats['train'][class_name],
                    'val': self.split_stats['val'][class_name],
                    'test': self.split_stats['test'][class_name],
                    'total': (self.split_stats['train'][class_name] +
                             self.split_stats['val'][class_name] +
                             self.split_stats['test'][class_name])
                }
                for class_name in self.classes
            },
            'total_statistics': total_stats
        }

        with open(stats_file, 'w') as f:
            json.dump(stats_data, f, indent=2)

        print(f"\n💾 Statistics saved to: {stats_file}")


def main():
    """
    Main execution function
    """
    # Configuration
    INPUT_DIR = "E:/Semisters/brain tumer prediction/brain_mri_10class_augmented"  # Preprocessed dataset
    OUTPUT_DIR = "data"  # Output directory for train/val/test splits

    # Split ratios
    TRAIN_RATIO = 0.70  # 70%
    VAL_RATIO = 0.15    # 15%
    TEST_RATIO = 0.15   # 15%

    # Random seed for reproducibility
    RANDOM_SEED = 42

    print("\n🚀 Starting Train/Val/Test Split\n")

    # Check if input directory exists
    if not os.path.exists(INPUT_DIR):
        print(f"❌ Error: Input directory '{INPUT_DIR}' not found!")
        print(f"Please run preprocess_brain_mri.py first to create the preprocessed dataset")
        return

    # Create splitter
    splitter = DatasetSplitter(
        input_dir=INPUT_DIR,
        output_dir=OUTPUT_DIR,
        train_ratio=TRAIN_RATIO,
        val_ratio=VAL_RATIO,
        test_ratio=TEST_RATIO,
        random_seed=RANDOM_SEED
    )

    # Perform split
    splitter.split_dataset()

    print("\n✅ Split completed!")
    print(f"📂 Split data saved to: {OUTPUT_DIR}")
    print("\n⚠️  IMPORTANT: DO NOT TOUCH THE TEST SET until final evaluation!")
    print("\n🔄 Next step: Start training your model with the train and val sets")


if __name__ == "__main__":
    main()


🚀 Starting Train/Val/Test Split

🔀 TRAIN/VAL/TEST SPLIT
Input directory: E:\Semisters\brain tumer prediction\brain_mri_10class_augmented
Output directory: data
Split ratios - Train: 70% | Val: 15% | Test: 15%
Random seed: 42

📁 Creating directory structure...

📊 Processing class: Astrocytoma
  Total images: 2872
  Copying to train: 2010 images
  Copying to val: 430 images
  Copying to test: 432 images

📊 Processing class: Ependymoma
  Total images: 3700
  Copying to train: 2590 images
  Copying to val: 555 images
  Copying to test: 555 images

📊 Processing class: Glioblastoma
  Total images: 3592
  Copying to train: 2514 images
  Copying to val: 538 images
  Copying to test: 540 images

📊 Processing class: Germinoma
  Total images: 3800
  Copying to train: 2660 images
  Copying to val: 570 images
  Copying to test: 570 images

📊 Processing class: Medulloblastoma
  Total images: 3738
  Copying to train: 2616 images
  Copying to val: 560 images
  Copying to test: 562 images

📊 Processin

In [3]:
"""
Dataset Analysis and Visualization Utilities
Author: Your Name
Project: 10-Class Brain Tumor Detection

This script provides utilities to:
1. Analyze class distribution
2. Visualize sample images
3. Check image quality
4. Generate reports
"""

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
import json


class DatasetAnalyzer:
    """
    Analyze and visualize brain MRI dataset
    """

    def __init__(self, dataset_dir):
        """
        Initialize the analyzer

        Args:
            dataset_dir (str): Path to dataset directory
        """
        self.dataset_dir = Path(dataset_dir)

        # Define the 10 classes
        self.classes = [
            'Astrocytoma',
            'Ependymoma',
            'Glioblastoma',
            'Germinoma',
            'Medulloblastoma',
            'Meningioma',
            'Oligodendroglioma',
            'Pituitary',
            'Schwannoma',
            'No_Tumor'
        ]

    def count_images_per_class(self):
        """
        Count number of images in each class

        Returns:
            dict: Dictionary with class counts
        """
        class_counts = {}

        for class_name in self.classes:
            class_dir = self.dataset_dir / class_name

            if not class_dir.exists():
                class_counts[class_name] = 0
                continue

            # Count image files
            image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff']
            count = 0
            for ext in image_extensions:
                count += len(list(class_dir.glob(f'*{ext}')))
                count += len(list(class_dir.glob(f'*{ext.upper()}')))

            class_counts[class_name] = count

        return class_counts

    def plot_class_distribution(self, save_path=None):
        """
        Plot class distribution bar chart

        Args:
            save_path (str): Path to save the plot (optional)
        """
        class_counts = self.count_images_per_class()

        # Create bar plot
        plt.figure(figsize=(12, 6))
        classes = list(class_counts.keys())
        counts = list(class_counts.values())

        colors = plt.cm.Set3(range(len(classes)))
        bars = plt.bar(classes, counts, color=colors, edgecolor='black', linewidth=1.2)

        # Add value labels on bars
        for bar, count in zip(bars, counts):
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height,
                    f'{int(count)}',
                    ha='center', va='bottom', fontsize=10, fontweight='bold')

        plt.xlabel('Tumor Class', fontsize=12, fontweight='bold')
        plt.ylabel('Number of Images', fontsize=12, fontweight='bold')
        plt.title('Brain Tumor MRI Dataset - Class Distribution', fontsize=14, fontweight='bold')
        plt.xticks(rotation=45, ha='right')
        plt.grid(axis='y', alpha=0.3, linestyle='--')
        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"📊 Plot saved to: {save_path}")

        plt.show()

    def analyze_image_properties(self, sample_size=None):
        """
        Analyze image properties (size, aspect ratio, etc.)

        Args:
            sample_size (int): Number of images to sample per class (None = all)

        Returns:
            dict: Statistics about image properties
        """
        image_sizes = []
        aspect_ratios = []

        for class_name in self.classes:
            class_dir = self.dataset_dir / class_name

            if not class_dir.exists():
                continue

            # Get image files
            image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff']
            image_files = []
            for ext in image_extensions:
                image_files.extend(list(class_dir.glob(f'*{ext}')))
                image_files.extend(list(class_dir.glob(f'*{ext.upper()}')))

            # Sample if needed
            if sample_size and len(image_files) > sample_size:
                image_files = np.random.choice(image_files, sample_size, replace=False)

            # Analyze each image
            for img_path in image_files:
                try:
                    img = cv2.imread(str(img_path))
                    if img is not None:
                        h, w = img.shape[:2]
                        image_sizes.append((w, h))
                        aspect_ratios.append(w / h)
                except Exception as e:
                    print(f"Error reading {img_path}: {e}")

        if not image_sizes:
            return None

        # Calculate statistics
        widths = [size[0] for size in image_sizes]
        heights = [size[1] for size in image_sizes]

        stats = {
            'total_analyzed': len(image_sizes),
            'width': {
                'min': min(widths),
                'max': max(widths),
                'mean': np.mean(widths),
                'median': np.median(widths)
            },
            'height': {
                'min': min(heights),
                'max': max(heights),
                'mean': np.mean(heights),
                'median': np.median(heights)
            },
            'aspect_ratio': {
                'min': min(aspect_ratios),
                'max': max(aspect_ratios),
                'mean': np.mean(aspect_ratios),
                'median': np.median(aspect_ratios)
            }
        }

        return stats

    def visualize_samples(self, n_samples=3, save_path=None):
        """
        Visualize random samples from each class

        Args:
            n_samples (int): Number of samples per class
            save_path (str): Path to save the visualization
        """
        n_classes = len(self.classes)
        fig, axes = plt.subplots(n_classes, n_samples, figsize=(4*n_samples, 3*n_classes))

        if n_classes == 1:
            axes = axes.reshape(1, -1)
        if n_samples == 1:
            axes = axes.reshape(-1, 1)

        for i, class_name in enumerate(self.classes):
            class_dir = self.dataset_dir / class_name

            if not class_dir.exists():
                for j in range(n_samples):
                    axes[i, j].text(0.5, 0.5, 'No images',
                                   ha='center', va='center', fontsize=12)
                    axes[i, j].axis('off')
                continue

            # Get image files
            image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff']
            image_files = []
            for ext in image_extensions:
                image_files.extend(list(class_dir.glob(f'*{ext}')))
                image_files.extend(list(class_dir.glob(f'*{ext.upper()}')))

            # Sample random images
            if len(image_files) >= n_samples:
                selected = np.random.choice(image_files, n_samples, replace=False)
            else:
                selected = image_files + [image_files[0]] * (n_samples - len(image_files))

            # Display images
            for j, img_path in enumerate(selected):
                try:
                    img = cv2.imread(str(img_path))
                    if img is not None:
                        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                        axes[i, j].imshow(img_rgb)
                    else:
                        axes[i, j].text(0.5, 0.5, 'Failed to load',
                                       ha='center', va='center', fontsize=10)
                except Exception as e:
                    axes[i, j].text(0.5, 0.5, f'Error: {str(e)[:20]}',
                                   ha='center', va='center', fontsize=8)

                axes[i, j].axis('off')

                if j == 0:
                    axes[i, j].set_title(f'{class_name}', fontsize=12, fontweight='bold', loc='left')

        plt.suptitle('Sample Images from Each Class', fontsize=16, fontweight='bold', y=0.995)
        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"🖼️  Visualization saved to: {save_path}")

        plt.show()

    def generate_report(self, output_file='dataset_report.txt'):
        """
        Generate a comprehensive text report

        Args:
            output_file (str): Path to save the report
        """
        class_counts = self.count_images_per_class()
        total_images = sum(class_counts.values())

        report = []
        report.append("=" * 80)
        report.append("BRAIN TUMOR MRI DATASET ANALYSIS REPORT")
        report.append("=" * 80)
        report.append(f"Dataset Directory: {self.dataset_dir}")
        report.append(f"Total Images: {total_images}")
        report.append(f"Number of Classes: {len(self.classes)}")
        report.append("\n" + "-" * 80)
        report.append("CLASS DISTRIBUTION")
        report.append("-" * 80)

        for class_name, count in sorted(class_counts.items(), key=lambda x: x[1], reverse=True):
            percentage = (count / total_images * 100) if total_images > 0 else 0
            report.append(f"{class_name:20s}: {count:6d} images ({percentage:5.2f}%)")

        # Analyze image properties
        print("\n🔍 Analyzing image properties (sampling images)...")
        stats = self.analyze_image_properties(sample_size=100)

        if stats:
            report.append("\n" + "-" * 80)
            report.append("IMAGE PROPERTIES (based on sample)")
            report.append("-" * 80)
            report.append(f"Total analyzed: {stats['total_analyzed']}")
            report.append(f"\nWidth  - Min: {stats['width']['min']}, Max: {stats['width']['max']}, "
                         f"Mean: {stats['width']['mean']:.1f}, Median: {stats['width']['median']:.1f}")
            report.append(f"Height - Min: {stats['height']['min']}, Max: {stats['height']['max']}, "
                         f"Mean: {stats['height']['mean']:.1f}, Median: {stats['height']['median']:.1f}")
            report.append(f"Aspect Ratio - Min: {stats['aspect_ratio']['min']:.3f}, "
                         f"Max: {stats['aspect_ratio']['max']:.3f}, "
                         f"Mean: {stats['aspect_ratio']['mean']:.3f}")

        report.append("\n" + "=" * 80)

        # Save report
        report_text = "\n".join(report)

        output_path = self.dataset_dir / output_file
        with open(output_path, 'w') as f:
            f.write(report_text)

        print(report_text)
        print(f"\n📄 Report saved to: {output_path}")


def main():
    """
    Main execution function
    """
    import sys

    # Get dataset directory from command line or use default
    if len(sys.argv) > 1:
        dataset_dir = sys.argv[1]
    else:
        dataset_dir = "E:/Semisters/brain tumer prediction/brain_mri_10class_augmented"  # Default to processed dataset

    print(f"\n🔍 Analyzing dataset: {dataset_dir}\n")

    if not os.path.exists(dataset_dir):
        print(f"❌ Error: Directory '{dataset_dir}' not found!")
        print("\nUsage: python analyze_dataset.py [dataset_directory]")
        return

    # Create analyzer
    analyzer = DatasetAnalyzer(dataset_dir)

    # Generate report
    print("📊 Generating dataset report...\n")
    analyzer.generate_report()

    # Plot class distribution
    print("\n📈 Creating class distribution plot...")
    analyzer.plot_class_distribution(save_path=f"{dataset_dir}/class_distribution.png")

    # Visualize samples
    print("\n🖼️  Creating sample visualization...")
    analyzer.visualize_samples(n_samples=3, save_path=f"{dataset_dir}/sample_images.png")

    print("\n✅ Analysis completed!")


if __name__ == "__main__":
    main()


🔍 Analyzing dataset: --f="c:\Users\FAIZ SIDDIQUI\AppData\Roaming\jupyter\runtime\kernel-v3569c05e50e8c7d5058a205744edc751e11ade859.json"

❌ Error: Directory '--f="c:\Users\FAIZ SIDDIQUI\AppData\Roaming\jupyter\runtime\kernel-v3569c05e50e8c7d5058a205744edc751e11ade859.json"' not found!

Usage: python analyze_dataset.py [dataset_directory]
